In [1]:
# Importing libraries
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import os
import numpy as np
import torch.nn as nn
import torch.nn.functional as F


In [2]:
# COnstants
NUM_DISKS = 4
NUM_PEGS = 3
MAX_EPOCHS = 1000
torch.manual_seed(4)

In [3]:
class ImageDataset(Dataset):
    """
    -------------------------------------------------------
    Load images from a directory and apply transformations.
    -------------------------------------------------------
    Parameters:
        image_dir (str): Directory with all the images.
        transform (callable, optional): Optional transform to be applied on an image.
        target_transform (callable): Should convert the target to a tensor.
        num_repeats (int): Number of times to repeat the dataset.
    -------------------------------------------------------
    """
    def __init__(self, image_dir, target_transform=None, transform=None, num_repeats=10):
        self.image_dir = image_dir
        self.num_repeats = num_repeats
        self.image_filenames = [f for f in os.listdir(image_dir) if f.endswith('.png') or f.endswith('.jpg')]
        self.box = (775, 472, 1630, 858)
        self.target_transform = target_transform
        self.transform = transform
    
    def __len__(self):
        """
        -------------------------------------------------------
        Returns the number of images in the dataset.
        -------------------------------------------------------
        Returns:
            len: Number of images in the dataset (int)
        -------------------------------------------------------
        """
        return len(self.image_filenames)*self.num_repeats
    
    def __getitem__(self, idx):
        """
        -------------------------------------------------------
        Load an image and its corresponding target.
        -------------------------------------------------------
        Parameters:
           idx : Index of the image to load (int)
        Returns:
            cropped_image: Transformed image (PIL Image)
            target: Target label (torch.Tensor)
        -------------------------------------------------------
        """
        idx = idx % len(self.image_filenames)  # Repeat the dataset
        # Load image
        img_name = self.image_filenames[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path)
        # Crop image
        cropped_image = image.crop(self.box)

        if self.transform:
            cropped_image = self.transform(cropped_image)

        label_str = img_name.split('_', 1)[1].rsplit('.', 1)[0]
        label_list = eval(label_str)
        if self.target_transform:
            label_list = self.target_transform(label_list)
        else:
            label_list = torch.tensor(label_list[::-1], dtype=torch.long)

        return cropped_image, label_list
    
def observation_to_state_matrix(observation):
    """
    -------------------------------------------------------
    Converts the observation from the environment to a state matrix.
    -------------------------------------------------------
    Parameters:
       observation - the observation from the environment (tuple)
    Returns:
         state_matrix - the state matrix (torch tensor)
    -------------------------------------------------------
    """
    assert len(observation) == NUM_DISKS, f"Invalid observation length: {len(observation)}"
    # Initialize a zero matrix of shape (num_disks, num_pegs)
    state_matrix = torch.zeros(NUM_PEGS, NUM_DISKS)

    # Set the appropriate column for each disk
    for disk, peg in enumerate(observation):
        state_matrix[peg, disk] = 1

    return state_matrix

def label_transform(label_list):
    """
    -------------------------------------------------------
    Transform the label list to a tensor.
    -------------------------------------------------------
    Parameters:
        label_list (list): List of labels to transform.
    Returns:
        label_tensor: Transformed label tensor (torch.Tensor)
    -------------------------------------------------------
    """
    new_state = label_list[::-1]  # Reverse the list
    return observation_to_state_matrix(new_state)

image_transform = T.Compose([
    T.Resize(224),  # Resize shortest edge to 224
    T.RandomVerticalFlip(p=0.5),  # 50% chance to flip vertically
    T.RandomRotation(degrees=30),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.RandomGrayscale(p=0.5),     # 50% chance to convert to grayscale
    T.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 2)),  # Random Gaussian blur
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [4]:
class CNNStateLearner(nn.Module):
    """
    -------------------------------------------------------
    CNN model for learning the state of the Tower of Hanoi game.
    -------------------------------------------------------
    Parameters:
        num_pegs (int): Number of pegs in the game.
        num_disks (int): Number of disks in the game.
    """
    def __init__(self, num_pegs, num_disks):
        super(CNNStateLearner, self).__init__()
        self.num_pegs = num_pegs
        self.num_disks = num_disks
        
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),  
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.linear_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 62, 512),  # Much smaller input size!
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, num_pegs * num_disks)
        )
        
    def forward(self, X):
        """
        -------------------------------------------------------
        Forward pass through the CNN model.
        -------------------------------------------------------
        Parameters:
            X - Input tensor (torch.Tensor)
        Returns:
            output - Output tensor (torch.Tensor)
        -------------------------------------------------------
        """
        # Pass through CNN layers
        X = self.cnn(X)
        # Pass through linear layers
        output = self.linear_layers(X)
        # Reshape output to (batch_size, num_pegs, num_disks)
        output = output.view(-1, self.num_pegs, self.num_disks)
        # probabilities = F.softmax(output, dim=-1)  # Apply softmax to get probabilities across pegs
        return output

model = CNNStateLearner(num_pegs=NUM_PEGS, num_disks=NUM_DISKS)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
ds = ImageDataset(
    image_dir='images',
    # target_transform=label_transform,
    transform=image_transform,
    num_repeats=5
)
data_loader = DataLoader(
    ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

In [6]:
for epoch in range(MAX_EPOCHS):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(data_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
    
    print(f'[Epoch {epoch + 1}] loss: {running_loss / len(data_loader):.3f}')

print('Finished Training')

[Epoch 1] loss: 1.103
[Epoch 2] loss: 1.100
[Epoch 3] loss: 1.097
[Epoch 4] loss: 1.092
[Epoch 5] loss: 1.078
[Epoch 6] loss: 1.064
[Epoch 7] loss: 1.026
[Epoch 8] loss: 0.988
[Epoch 9] loss: 0.969
[Epoch 10] loss: 0.941
[Epoch 11] loss: 0.932
[Epoch 12] loss: 0.927
[Epoch 13] loss: 0.891
[Epoch 14] loss: 0.878
[Epoch 15] loss: 0.854
[Epoch 16] loss: 0.830
[Epoch 17] loss: 0.799
[Epoch 18] loss: 0.777
[Epoch 19] loss: 0.768
[Epoch 20] loss: 0.741
[Epoch 21] loss: 0.732
[Epoch 22] loss: 0.701
[Epoch 23] loss: 0.684
[Epoch 24] loss: 0.661
[Epoch 25] loss: 0.656
[Epoch 26] loss: 0.639
[Epoch 27] loss: 0.608
[Epoch 28] loss: 0.582
[Epoch 29] loss: 0.604
[Epoch 30] loss: 0.554
[Epoch 31] loss: 0.572
[Epoch 32] loss: 0.540
[Epoch 33] loss: 0.515
[Epoch 34] loss: 0.510
[Epoch 35] loss: 0.511
[Epoch 36] loss: 0.507
[Epoch 37] loss: 0.498
[Epoch 38] loss: 0.466
[Epoch 39] loss: 0.433
[Epoch 40] loss: 0.461
[Epoch 41] loss: 0.451
[Epoch 42] loss: 0.434
[Epoch 43] loss: 0.435
[Epoch 44] loss: 0.4

KeyboardInterrupt: 

In [7]:
outputs.shape, labels.shape

(torch.Size([32, 3, 4]), torch.Size([32, 4]))

In [8]:
F.softmax(outputs, dim=1), labels

(tensor([[[1.9544e-07, 9.5215e-06, 4.0467e-06, 2.0013e-11],
          [9.1691e-07, 9.4327e-07, 1.0202e-05, 3.0484e-09],
          [1.0000e+00, 9.9999e-01, 9.9999e-01, 1.0000e+00]],
 
         [[4.6757e-11, 5.0494e-03, 1.0000e+00, 5.1257e-06],
          [5.5097e-15, 1.0842e-05, 1.8305e-06, 9.9999e-01],
          [1.0000e+00, 9.9494e-01, 3.2831e-08, 4.8095e-06]],
 
         [[4.8390e-10, 2.2525e-07, 1.1086e-14, 3.5782e-08],
          [1.0000e+00, 1.0000e+00, 1.0185e-07, 1.0000e+00],
          [1.1348e-11, 3.5097e-07, 1.0000e+00, 1.6819e-06]],
 
         [[2.6199e-05, 2.6318e-05, 1.4076e-04, 9.1276e-06],
          [1.9894e-06, 9.9956e-01, 7.6015e-02, 9.9951e-01],
          [9.9997e-01, 4.1639e-04, 9.2384e-01, 4.8318e-04]],
 
         [[1.0000e+00, 3.4655e-05, 9.9996e-01, 1.2633e-03],
          [3.3166e-06, 9.9996e-01, 6.0804e-08, 8.9727e-08],
          [8.2409e-08, 7.5845e-06, 3.5356e-05, 9.9874e-01]],
 
         [[4.3652e-04, 9.9159e-01, 1.7518e-08, 1.3599e-05],
          [9.9956e-01, 2.

In [9]:
PATH = './nn_models/state_extractor.pth'
torch.save(model.state_dict(), PATH)